# Optimisation Algorithms in Neural Networks: SGD vs RMSprop vs Adam

This notebook compares three optimisation algorithms used to train neural networks:

- **Stochastic Gradient Descent (SGD)**
- **RMSprop**
- **Adam**

The goal is to study how each optimiser affects:

- training loss
- convergence speed
- classification accuracy
- decision boundary shape

To keep the implementation transparent and easy to explain in a tutorial, the neural network in this notebook is implemented **from scratch in NumPy** rather than using a high-level deep-learning framework.


## 1. Imports

These libraries are used for data generation, preprocessing, visualisation, and evaluation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


## 2. Dataset

A synthetic `make_moons` dataset is used because it is non-linear and easy to visualise. This makes it suitable for comparing how optimisers influence training behaviour on the same model.


In [ ]:
X, y = make_moons(n_samples=600, noise=0.25, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

print('Training samples:', X_train.shape[0])
print('Test samples:', X_test.shape[0])


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train.ravel(), alpha=0.8)
plt.title('Training data (make_moons)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()


## 3. Activation and helper functions

The hidden layers use **ReLU**, while the output layer uses a **sigmoid** function for binary classification.


In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(float)

def sigmoid(z):
    z = np.clip(z, -50, 50)
    return 1 / (1 + np.exp(-z))

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-8
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


## 4. Build a simple MLP from scratch

The model used in all experiments has the same architecture:

- input layer: 2 features
- hidden layer 1: 16 neurons + ReLU
- hidden layer 2: 16 neurons + ReLU
- output layer: 1 neuron + sigmoid

Only the **optimiser** will change across experiments.


In [ ]:
class SimpleMLP:
    def __init__(self, input_dim=2, hidden1=16, hidden2=16, seed=42):
        rng = np.random.default_rng(seed)
        self.params = {
            'W1': rng.normal(scale=np.sqrt(2 / input_dim), size=(input_dim, hidden1)),
            'b1': np.zeros((1, hidden1)),
            'W2': rng.normal(scale=np.sqrt(2 / hidden1), size=(hidden1, hidden2)),
            'b2': np.zeros((1, hidden2)),
            'W3': rng.normal(scale=np.sqrt(2 / hidden2), size=(hidden2, 1)),
            'b3': np.zeros((1, 1)),
        }

    def forward(self, X):
        p = self.params
        z1 = X @ p['W1'] + p['b1']
        a1 = relu(z1)
        z2 = a1 @ p['W2'] + p['b2']
        a2 = relu(z2)
        z3 = a2 @ p['W3'] + p['b3']
        y_hat = sigmoid(z3)
        cache = (X, z1, a1, z2, a2, z3, y_hat)
        return y_hat, cache

    def backward(self, cache, y_true):
        X, z1, a1, z2, a2, z3, y_hat = cache
        m = len(X)

        dz3 = (y_hat - y_true) / m
        dW3 = a2.T @ dz3
        db3 = np.sum(dz3, axis=0, keepdims=True)

        da2 = dz3 @ self.params['W3'].T
        dz2 = da2 * relu_grad(z2)
        dW2 = a1.T @ dz2
        db2 = np.sum(dz2, axis=0, keepdims=True)

        da1 = dz2 @ self.params['W2'].T
        dz1 = da1 * relu_grad(z1)
        dW1 = X.T @ dz1
        db1 = np.sum(dz1, axis=0, keepdims=True)

        grads = {
            'W1': dW1, 'b1': db1,
            'W2': dW2, 'b2': db2,
            'W3': dW3, 'b3': db3,
        }
        return grads

    def predict(self, X):
        y_hat, _ = self.forward(X)
        return (y_hat >= 0.5).astype(int)


## 5. Optimisers

Three optimisation methods are implemented:

- **SGD**: updates weights directly using the gradient
- **RMSprop**: adapts learning rates using a moving average of squared gradients
- **Adam**: combines momentum and adaptive learning rates


In [ ]:
def train_model(optimizer_name, learning_rate, epochs=250):
    model = SimpleMLP(seed=42)
    params = model.params
    history = {'loss': []}

    if optimizer_name == 'RMSprop':
        s = {k: np.zeros_like(v) for k, v in params.items()}
        beta = 0.9
        eps = 1e-8

    if optimizer_name == 'Adam':
        m = {k: np.zeros_like(v) for k, v in params.items()}
        v = {k: np.zeros_like(v) for k, v in params.items()}
        beta1, beta2 = 0.9, 0.999
        eps = 1e-8

    for t in range(1, epochs + 1):
        y_hat, cache = model.forward(X_train)
        loss = binary_cross_entropy(y_train, y_hat)
        grads = model.backward(cache, y_train)

        for key in params:
            g = grads[key]

            if optimizer_name == 'SGD':
                params[key] -= learning_rate * g

            elif optimizer_name == 'RMSprop':
                s[key] = beta * s[key] + (1 - beta) * (g ** 2)
                params[key] -= learning_rate * g / (np.sqrt(s[key]) + eps)

            elif optimizer_name == 'Adam':
                m[key] = beta1 * m[key] + (1 - beta1) * g
                v[key] = beta2 * v[key] + (1 - beta2) * (g ** 2)
                m_hat = m[key] / (1 - beta1 ** t)
                v_hat = v[key] / (1 - beta2 ** t)
                params[key] -= learning_rate * m_hat / (np.sqrt(v_hat) + eps)

            else:
                raise ValueError('Unknown optimiser name')

        history['loss'].append(loss)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    return model, history, train_acc, test_acc


## 6. Run the experiments

Only the optimiser changes. The architecture, dataset, preprocessing, and number of epochs stay the same.


In [ ]:
settings = {
    'SGD': 0.08,
    'RMSprop': 0.01,
    'Adam': 0.01,
}

results = {}
for name, lr in settings.items():
    model, history, train_acc, test_acc = train_model(name, lr, epochs=250)
    results[name] = {
        'model': model,
        'history': history,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'learning_rate': lr,
    }

summary = pd.DataFrame([
    {
        'Optimizer': name,
        'Learning Rate': values['learning_rate'],
        'Train Accuracy': values['train_acc'],
        'Test Accuracy': values['test_acc'],
        'Final Loss': values['history']['loss'][-1],
    }
    for name, values in results.items()
])

summary.sort_values('Test Accuracy', ascending=False)


## 7. Loss curve comparison

This plot shows how quickly each optimiser reduces the training loss over epochs.


In [ ]:
plt.figure(figsize=(8, 5))
for name, values in results.items():
    plt.plot(values['history']['loss'], label=name)
plt.xlabel('Epoch')
plt.ylabel('Training loss')
plt.title('Loss curve comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 8. Accuracy comparison

This bar chart makes it easier to compare final test accuracy across the three optimisation methods.


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(summary['Optimizer'], summary['Test Accuracy'])
plt.ylim(0, 1)
plt.ylabel('Test accuracy')
plt.title('Test accuracy by optimiser')
plt.show()


## 9. Decision boundaries

These plots show how the trained network separates the two classes after being trained with each optimiser.


In [ ]:
def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.35)
    plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), edgecolor='k', s=25)
    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ['SGD', 'RMSprop', 'Adam']):
    plt.sca(ax)
    plot_decision_boundary(results[name]['model'], X_test, y_test, f'{name} decision boundary')
plt.tight_layout()
plt.show()


## 10. Interpretation

After running the notebook, pay attention to the following:

- Which optimiser reduces loss fastest?
- Which optimiser produces the smoothest convergence curve?
- Do higher training speeds translate into better test accuracy?
- Does the decision boundary become cleaner or more stable with adaptive optimisers?

These observations can be used directly in the tutorial report, especially in the experimental analysis section.


## 11. References to mention in the report/notebook

- Goodfellow, Bengio, and Courville — *Deep Learning*
- Bishop — *Pattern Recognition and Machine Learning*
- Kingma and Ba (2015) — Adam optimiser paper
- Hinton et al. lecture notes / RMSprop explanations
